# 3회차 실습 — Custom Tool 개발과 AI Agent 구현

## 학습목표
- LLM, Tool, Agent의 역할 차이를 설명한다.
- Python 함수로 Custom Tool을 직접 만든다.
- LLM이 사용자 요청을 분석해 적절한 Tool을 선택하도록 구현한다.
- Tool 실행 결과를 다시 LLM에 전달해 최종 응답을 생성한다.
- Agent 실행 과정을 로그로 확인하고 오류 상황을 처리한다.

### 실행 환경
- 🟢 **LOCAL**: 로컬 Jupyter에서 실행 가능
- 🔴 **GPU**: Qwen3-8B 추론을 위해 CUDA GPU 필요
- 📸 **CAPTURE**: 강의자료 캡처 권장

> 이번 차시의 핵심 흐름  
> **사용자 질문 → LLM 판단 → Tool 선택 → Python 함수 실행 → Tool 결과 → LLM 최종 응답**


## Step 1. LLM · Tool · Agent 이해 — 🟢 LOCAL

- **LLM**: 자연어를 이해하고 응답을 생성하는 두뇌
- **Tool**: 계산, 문자열 처리, 시간 확인 등 실제 작업을 수행하는 함수
- **Agent**: LLM의 판단과 Tool 실행을 연결하여 목표를 수행하는 제어 흐름

이번 실습에서는 외부 API 대신 **로컬 Python 함수**를 Tool로 사용합니다.


In [1]:
import sys
import json
import re
import math
from datetime import datetime

print("Python :", sys.version.split()[0])
print("기본 라이브러리 준비 완료")


Python : 3.12.3
기본 라이브러리 준비 완료


## Step 2. 첫 번째 Custom Tool — 계산기 — 🟢 LOCAL

`eval()`을 사용하지 않고 허용된 연산자만 처리하는 간단한 계산기를 구현합니다.


In [2]:
import ast
import operator as op

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.FloorDiv: op.floordiv,
    ast.Mod: op.mod,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.UAdd: op.pos,
}

def _safe_eval_node(node):
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_eval_node(node.left)
        right = _safe_eval_node(node.right)
        return _ALLOWED_OPERATORS[type(node.op)](left, right)

    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval_node(node.operand))

    raise ValueError("허용되지 않은 수식입니다.")

def calculator(expression: str):
    """사칙연산과 일부 기본 연산을 안전하게 계산한다."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval_node(tree)
        return {"ok": True, "result": result}
    except Exception as e:
        return {"ok": False, "error": str(e)}

print(calculator("125 * 8 + 30"))
print(calculator("(100 - 20) / 4"))


{'ok': True, 'result': 1030}
{'ok': True, 'result': 20.0}


## Step 3. 추가 Custom Tool 만들기 — 🟢 LOCAL

이번 실습에서는 세 가지 Tool을 사용합니다.

1. `calculator` : 수식 계산
2. `text_length` : 문자열의 글자 수 계산
3. `current_time` : 현재 시스템 시각 확인


In [3]:
def text_length(text: str):
    """입력 문자열의 글자 수를 반환한다."""
    return {
        "ok": True,
        "text": text,
        "length": len(text)
    }

def current_time():
    """현재 시스템의 날짜와 시각을 반환한다."""
    now = datetime.now()
    return {
        "ok": True,
        "datetime": now.strftime("%Y-%m-%d %H:%M:%S")
    }

print(text_length("Yeungnam University AI PBL"))
print(current_time())


{'ok': True, 'text': 'Yeungnam University AI PBL', 'length': 26}
{'ok': True, 'datetime': '2026-10-05 06:26:27'}


## Step 4. Tool Registry 구성 — 🟢 LOCAL / 📸

Agent가 Tool 이름을 통해 실제 Python 함수를 찾을 수 있도록 Registry를 만듭니다.


In [4]:
TOOL_REGISTRY = {
    "calculator": calculator,
    "text_length": text_length,
    "current_time": current_time,
}

TOOL_DESCRIPTIONS = {
    "calculator": {
        "description": "수학 수식을 계산합니다.",
        "arguments": {"expression": "계산할 수식 문자열"}
    },
    "text_length": {
        "description": "문자열의 글자 수를 계산합니다.",
        "arguments": {"text": "길이를 확인할 문자열"}
    },
    "current_time": {
        "description": "현재 시스템 날짜와 시각을 확인합니다.",
        "arguments": {}
    }
}

print("사용 가능한 Tool")
for name, spec in TOOL_DESCRIPTIONS.items():
    print(f"- {name}: {spec['description']}")


사용 가능한 Tool
- calculator: 수학 수식을 계산합니다.
- text_length: 문자열의 글자 수를 계산합니다.
- current_time: 현재 시스템 날짜와 시각을 확인합니다.


## Step 5. Tool 실행기 만들기 — 🟢 LOCAL

LLM이 선택한 Tool 이름과 arguments를 받아 실제 Python 함수를 실행합니다.
존재하지 않는 Tool이나 잘못된 arguments도 오류로 처리합니다.


In [5]:
def execute_tool(tool_name: str, arguments: dict):
    if tool_name not in TOOL_REGISTRY:
        return {
            "ok": False,
            "error": f"존재하지 않는 Tool: {tool_name}"
        }

    if not isinstance(arguments, dict):
        return {
            "ok": False,
            "error": "arguments는 dict 형식이어야 합니다."
        }

    try:
        return TOOL_REGISTRY[tool_name](**arguments)
    except TypeError as e:
        return {"ok": False, "error": f"인자 오류: {e}"}
    except Exception as e:
        return {"ok": False, "error": str(e)}

print(execute_tool("calculator", {"expression": "33 * 30"}))
print(execute_tool("text_length", {"text": "AI Agent"}))
print(execute_tool("current_time", {}))
print(execute_tool("unknown_tool", {}))


{'ok': True, 'result': 990}
{'ok': True, 'text': 'AI Agent', 'length': 8}
{'ok': True, 'datetime': '2026-10-05 06:26:37'}
{'ok': False, 'error': '존재하지 않는 Tool: unknown_tool'}


## Step 6. Agent가 사용할 JSON 규약 — 🟢 LOCAL

LLM은 다음 두 형식 중 하나만 출력하도록 지시합니다.

### Tool이 필요한 경우
```json
{
  "action": "tool",
  "tool_name": "calculator",
  "arguments": {"expression": "33 * 30"}
}
```

### Tool이 필요 없는 경우
```json
{
  "action": "answer",
  "answer": "일반적인 설명 응답"
}
```

Agent는 이 JSON을 읽어 다음 행동을 결정합니다.


## Step 7. Qwen3-8B 환경 확인 — 🔴 GPU

로컬에서는 여기까지 작성·테스트할 수 있습니다.  
이제부터 LLM 판단이 필요한 셀은 RunPod A100에서 실행합니다.


In [7]:
%pip install -U transformers accelerate huggingface_hub sentencepiece bitsandbytes pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 226.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 681.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 499.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 807.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 386.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 247.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 544.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.8/807.8 kB 380.1 MB/s  0:00:00
  Attempting uninstall: idna0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/19 [regex]
    Found existing installation: idna 3.10━━━━━━━━━━━━━━━━━━━━  4/19 [regex]
    Uninstalling idna-3.10:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/19 [regex]
      Successfully uninstalled idna-3.10━━━━━━━━━━━━━━━━━━━━━━  4/19 [regex]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [transformers] [transformers]ub]


In [8]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU 없음 — 이후 LLM 셀은 RunPod에서 실행하세요.")


PyTorch : 2.8.0+cu128
Transformers : 5.18.0
CUDA available : True
GPU : NVIDIA A100 80GB PCIe


In [17]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-8B",
    cache_dir="/workspace/.cache/huggingface/hub"
)

print("Qwen3-8B 다운로드 완료")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/d46195ac87f837ad233d02b2f80f148bf7c005e0.372ae8f4.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/d46195ac87f837ad233d02b2f80f148bf7c005e0.372ae8f4.incomplete'.
Continuing without setting permissions.
Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/20a8a9156fc8c3f25295ca067f61fdf120d517c5.668c9064.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/20a8a9156fc8c3f25295ca067f61fdf120d517c5.668c9064.incomplete'.
Continuing without setting permissions.
Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/31349551d90c7606f325fe0f11bbb8bd5fa0d7c7.d4534c6e.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hu

Qwen3-8B 다운로드 완료


## Step 8. Qwen3-8B 로드 — 🔴 GPU

2회차에서 사용한 Qwen3-8B를 Agent의 판단 모델로 사용합니다.


In [18]:
if not torch.cuda.is_available():
    raise RuntimeError("이 셀은 CUDA GPU 환경에서 실행해야 합니다.")

MODEL_ID = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

print("Model :", MODEL_ID)
print("GPU :", torch.cuda.get_device_name(0))
print(
    "VRAM Allocated :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model : Qwen/Qwen3-8B
GPU : NVIDIA A100 80GB PCIe
VRAM Allocated : 15.26 GB


## Step 9. LLM 호출 함수 — 🔴 GPU

Qwen3의 Thinking Mode를 끄고, Agent 제어에 사용할 짧고 구조화된 출력을 생성합니다.


In [19]:
def generate_llm(messages, max_new_tokens=300):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    input_tokens = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    ).strip()

    return response


## Step 10. Tool 선택 프롬프트 만들기 — 🔴 GPU

Tool 목록과 출력 규칙을 System Prompt에 명시합니다.


In [20]:
TOOL_SYSTEM_PROMPT = f"""
너는 Tool을 사용할 수 있는 AI Agent다.

사용 가능한 Tool:
{json.dumps(TOOL_DESCRIPTIONS, ensure_ascii=False, indent=2)}

규칙:
1. 계산이 필요하면 calculator를 사용한다.
2. 문자열 글자 수가 필요하면 text_length를 사용한다.
3. 현재 날짜/시각이 필요하면 current_time을 사용한다.
4. Tool이 필요하지 않은 일반 설명 질문은 직접 답한다.
5. 반드시 JSON 하나만 출력한다. Markdown 코드블록은 사용하지 않는다.

Tool이 필요한 경우:
{{
  "action": "tool",
  "tool_name": "도구이름",
  "arguments": {{...}}
}}

Tool이 필요 없는 경우:
{{
  "action": "answer",
  "answer": "최종 답변"
}}
"""

print(TOOL_SYSTEM_PROMPT)



너는 Tool을 사용할 수 있는 AI Agent다.

사용 가능한 Tool:
{
  "calculator": {
    "description": "수학 수식을 계산합니다.",
    "arguments": {
      "expression": "계산할 수식 문자열"
    }
  },
  "text_length": {
    "description": "문자열의 글자 수를 계산합니다.",
    "arguments": {
      "text": "길이를 확인할 문자열"
    }
  },
  "current_time": {
    "description": "현재 시스템 날짜와 시각을 확인합니다.",
    "arguments": {}
  }
}

규칙:
1. 계산이 필요하면 calculator를 사용한다.
2. 문자열 글자 수가 필요하면 text_length를 사용한다.
3. 현재 날짜/시각이 필요하면 current_time을 사용한다.
4. Tool이 필요하지 않은 일반 설명 질문은 직접 답한다.
5. 반드시 JSON 하나만 출력한다. Markdown 코드블록은 사용하지 않는다.

Tool이 필요한 경우:
{
  "action": "tool",
  "tool_name": "도구이름",
  "arguments": {...}
}

Tool이 필요 없는 경우:
{
  "action": "answer",
  "answer": "최종 답변"
}



## Step 11. JSON 파싱 함수 — 🟢 LOCAL

LLM 출력에 코드펜스가 붙는 경우를 대비해 최소한의 정리 후 JSON으로 변환합니다.


In [21]:
def parse_agent_json(text: str):
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {
            "action": "error",
            "error": f"JSON 파싱 실패: {e}",
            "raw_output": text
        }


## Step 12. Tool 선택 테스트 — 🔴 GPU / 📸

먼저 Agent 전체를 만들기 전에 LLM이 올바른 Tool을 선택하는지 확인합니다.


In [22]:
test_questions = [
    "125 곱하기 8에 30을 더하면 얼마야?",
    "'GPU 기반 AI PBL'의 글자 수를 알려줘.",
    "지금 날짜와 시간을 알려줘.",
    "LLM과 Agent의 차이를 한 문장으로 설명해줘."
]

for question in test_questions:
    messages = [
        {"role": "system", "content": TOOL_SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
    raw = generate_llm(messages)
    parsed = parse_agent_json(raw)

    print("=" * 70)
    print("질문 :", question)
    print("LLM 원본 :", raw)
    print("파싱 결과 :", parsed)


질문 : 125 곱하기 8에 30을 더하면 얼마야?
LLM 원본 : {
  "action": "tool",
  "tool_name": "calculator",
  "arguments": {
    "expression": "125 * 8 + 30"
  }
}
파싱 결과 : {'action': 'tool', 'tool_name': 'calculator', 'arguments': {'expression': '125 * 8 + 30'}}
질문 : 'GPU 기반 AI PBL'의 글자 수를 알려줘.
LLM 원본 : {
  "action": "tool",
  "tool_name": "text_length",
  "arguments": {
    "text": "GPU 기반 AI PBL"
  }
}
파싱 결과 : {'action': 'tool', 'tool_name': 'text_length', 'arguments': {'text': 'GPU 기반 AI PBL'}}
질문 : 지금 날짜와 시간을 알려줘.
LLM 원본 : {
  "action": "tool",
  "tool_name": "current_time",
  "arguments": {}
}
파싱 결과 : {'action': 'tool', 'tool_name': 'current_time', 'arguments': {}}
질문 : LLM과 Agent의 차이를 한 문장으로 설명해줘.
LLM 원본 : LLM은 대규모 언어 모델로, 주어진 입력에 따라 텍스트를 생성하거나 이해하는 역할을 하지만, Agent는 특정 목표를 달성하기 위해 도구를 사용하고 환경과 상호작용하는 능력이 있는 시스템입니다.
파싱 결과 : {'action': 'error', 'error': 'JSON 파싱 실패: Expecting value: line 1 column 1 (char 0)', 'raw_output': 'LLM은 대규모 언어 모델로, 주어진 입력에 따라 텍스트를 생성하거나 이해하는 역할을 하지만, Agent는 특정 목표를 달성하기 위해 도구를

## Step 13. Tool 결과를 최종 응답으로 바꾸는 함수 — 🔴 GPU

Tool 실행 결과를 그대로 보여주는 대신, 사용자 질문과 Tool 결과를 LLM에 다시 전달해 자연어 최종 응답을 생성합니다.


In [23]:
def make_final_answer(user_query: str, tool_name: str, tool_result: dict):
    messages = [
        {
            "role": "system",
            "content": (
                "사용자의 질문과 Tool 실행 결과를 바탕으로 "
                "간결하고 정확한 한국어 최종 답변을 작성해라. "
                "Tool 결과에 없는 사실은 만들지 마라."
            )
        },
        {
            "role": "user",
            "content": (
                f"사용자 질문: {user_query}\n"
                f"사용한 Tool: {tool_name}\n"
                f"Tool 결과: {json.dumps(tool_result, ensure_ascii=False)}"
            )
        }
    ]
    return generate_llm(messages, max_new_tokens=200)


## Step 14. 단일 Tool AI Agent 구현 — 🔴 GPU / 📸

이제 **판단 → Tool 실행 → 최종 응답**을 하나의 함수로 연결합니다.


In [24]:
def run_agent(user_query: str, verbose=True):
    # 1) LLM이 Tool 사용 여부와 Tool을 결정
    decision_messages = [
        {"role": "system", "content": TOOL_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    raw_decision = generate_llm(decision_messages)
    decision = parse_agent_json(raw_decision)

    if verbose:
        print("[1] 사용자 질문")
        print(user_query)
        print("\n[2] LLM 판단")
        print(decision)

    # 2) 일반 답변
    if decision.get("action") == "answer":
        return {
            "status": "success",
            "used_tool": False,
            "decision": decision,
            "final_answer": decision.get("answer", "")
        }

    # 3) JSON 오류
    if decision.get("action") == "error":
        return {
            "status": "error",
            "used_tool": False,
            "decision": decision,
            "final_answer": "Agent의 구조화 출력을 해석하지 못했습니다."
        }

    # 4) Tool 실행
    if decision.get("action") == "tool":
        tool_name = decision.get("tool_name")
        arguments = decision.get("arguments", {})

        tool_result = execute_tool(tool_name, arguments)

        if verbose:
            print("\n[3] Tool 실행")
            print("Tool :", tool_name)
            print("Arguments :", arguments)
            print("Result :", tool_result)

        # 5) Tool 실패
        if not tool_result.get("ok", False):
            return {
                "status": "tool_error",
                "used_tool": True,
                "tool_name": tool_name,
                "tool_result": tool_result,
                "final_answer": f"Tool 실행 오류: {tool_result.get('error')}"
            }

        # 6) Tool 결과를 이용해 최종 답변 생성
        final_answer = make_final_answer(
            user_query,
            tool_name,
            tool_result
        )

        if verbose:
            print("\n[4] 최종 응답")
            print(final_answer)

        return {
            "status": "success",
            "used_tool": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": tool_result,
            "decision": decision,
            "final_answer": final_answer
        }

    return {
        "status": "error",
        "used_tool": False,
        "decision": decision,
        "final_answer": "지원하지 않는 Agent action입니다."
    }


## Step 15. Agent 실행 실습 — 🔴 GPU / 📸

서로 다른 유형의 질문으로 Agent의 Tool 선택과 실행 과정을 확인합니다.


In [25]:
agent_questions = [
    "33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?",
    "'Custom Tool 개발과 AI Agent 구현'의 글자 수를 알려줘.",
    "현재 날짜와 시간을 알려줘.",
    "AI Agent가 무엇인지 한 문장으로 설명해줘."
]

agent_results = []

for q in agent_questions:
    print("\n" + "=" * 80)
    result = run_agent(q, verbose=True)
    agent_results.append(result)



[1] 사용자 질문
33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?

[2] LLM 판단
{'action': 'calculator', 'tool_name': 'calculator', 'arguments': {'expression': '33 * 30'}}

[1] 사용자 질문
'Custom Tool 개발과 AI Agent 구현'의 글자 수를 알려줘.

[2] LLM 판단
{'action': 'tool', 'tool_name': 'text_length', 'arguments': {'text': 'Custom Tool 개발과 AI Agent 구현'}}

[3] Tool 실행
Tool : text_length
Arguments : {'text': 'Custom Tool 개발과 AI Agent 구현'}
Result : {'ok': True, 'text': 'Custom Tool 개발과 AI Agent 구현', 'length': 27}

[4] 최종 응답
"Custom Tool 개발과 AI Agent 구현"의 글자 수는 27입니다.

[1] 사용자 질문
현재 날짜와 시간을 알려줘.

[2] LLM 판단
{'action': 'tool', 'tool_name': 'current_time', 'arguments': {}}

[3] Tool 실행
Tool : current_time
Arguments : {}
Result : {'ok': True, 'datetime': '2026-10-05 06:40:55'}

[4] 최종 응답
2026년 10월 5일 06시 40분 55초입니다.

[1] 사용자 질문
AI Agent가 무엇인지 한 문장으로 설명해줘.

[2] LLM 판단
{'action': 'error', 'error': 'JSON 파싱 실패: Expecting value: line 1 column 1 (char 0)', 'raw_output': 'AI Agent는 특정 작업을 수행하거나 특정 목표를 달성하기 위해 설계된 인공지능 시스템입니다.'}


## Step 16. Agent 실행 로그를 표로 정리 — 🟢 LOCAL / 📸

Agent의 판단과 Tool 사용 여부를 표 형태로 남기면 디버깅과 평가가 쉬워집니다.


In [26]:
import pandas as pd

log_rows = []

for q, result in zip(agent_questions, agent_results):
    log_rows.append({
        "question": q,
        "status": result.get("status"),
        "used_tool": result.get("used_tool"),
        "tool_name": result.get("tool_name"),
        "arguments": json.dumps(
            result.get("arguments", {}),
            ensure_ascii=False
        ),
        "final_answer": result.get("final_answer")
    })

agent_log_df = pd.DataFrame(log_rows)
agent_log_df


,question,status,used_tool,tool_name,arguments,final_answer
0,33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?,error,False,NaN,{},지원하지 않는 Agent action입니다.
1,'Custom Tool 개발과 AI Agent 구현'의 글자 수를 알려줘.,success,True,text_length,"{""text"": ""Custom Tool 개발과 AI Agent 구현""}","""Custom Tool 개발과 AI Agent 구현""의 글자 수는 27입니다."
2,현재 날짜와 시간을 알려줘.,success,True,current_time,{},2026년 10월 5일 06시 40분 55초입니다.
3,AI Agent가 무엇인지 한 문장으로 설명해줘.,error,False,NaN,{},Agent의 구조화 출력을 해석하지 못했습니다.


## Step 17. 오류 상황 테스트 — 🟢 LOCAL

Agent 서비스에서는 정상 동작뿐 아니라 **실패를 안전하게 처리하는 것**도 중요합니다.


In [27]:
error_tests = [
    ("없는 Tool", "weather", {}),
    ("잘못된 calculator 인자", "calculator", {"value": "10+20"}),
    ("허용하지 않는 수식", "calculator", {"expression": "__import__('os').system('echo test')"}),
]

for title, tool_name, args in error_tests:
    print("=" * 60)
    print(title)
    print(execute_tool(tool_name, args))


없는 Tool
{'ok': False, 'error': '존재하지 않는 Tool: weather'}
잘못된 calculator 인자
{'ok': False, 'error': "인자 오류: calculator() got an unexpected keyword argument 'value'"}
허용하지 않는 수식
{'ok': False, 'error': '허용되지 않은 수식입니다.'}


## Step 18. 학생 미션 — 새로운 Tool 추가 — 🟢 LOCAL + 🔴 GPU

팀별로 Custom Tool 하나를 추가합니다.

### 예시
- `word_count(text)` : 단어 수 계산
- `unit_converter(value, from_unit, to_unit)` : 단위 변환
- `keyword_counter(text, keyword)` : 특정 단어 출현 횟수
- `grade_average(scores)` : 평균 점수 계산

### 수행 순서
1. Python 함수 작성
2. `TOOL_REGISTRY` 등록
3. `TOOL_DESCRIPTIONS` 등록
4. Tool 선택 프롬프트에 설명 반영
5. Agent에게 질문하여 Tool 선택 확인
6. 실행 결과와 오류 상황 기록


## Step 19. 결과 저장 

Agent 실행 로그를 CSV로 저장합니다.


In [28]:
from pathlib import Path

results_dir = Path.cwd() / "results"
results_dir.mkdir(parents=True, exist_ok=True)

log_path = results_dir / "agent_tool_execution_log.csv"
agent_log_df.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료 :", log_path)


저장 완료 : /workspace/notebooks/results/agent_tool_execution_log.csv


## 3회차 실습 완료

### 핵심 산출물
- `05_Custom_Tool_Agent.ipynb`
- `results/agent_tool_execution_log.csv`
- 팀별 Custom Tool 1개 이상
- Tool 선택 및 실행 테스트 결과

### 이번 차시에서 완성한 구조

```text
사용자 질문
    ↓
Qwen3-8B
    ↓
Tool 필요 여부 판단
    ↓
Tool 선택 + arguments 생성
    ↓
Python Custom Tool 실행
    ↓
Tool Result
    ↓
Qwen3-8B 최종 응답
```

### 다음 차시 연결
다음 단계에서는 이 Agent 로직을 서비스 코드로 확장하여 **AI Agent 서비스 개발**로 연결합니다.
